# Day 10 - Spark SQL and Temp Views

**Topic:** Spark SQL and Temp Views  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึกใช้ `spark.sql`, `temp view`, `global temp view` และเปรียบเทียบ SQL API กับ DataFrame API ในงาน PySpark Lakehouse

วันนี้เราจะเรียนการใช้ Spark SQL ใน PySpark โดยเริ่มจากการสร้าง DataFrame แล้ว register เป็น temporary view เพื่อให้ query ด้วย SQL ได้ จากนั้นจะเทียบ logic เดียวกันระหว่าง SQL กับ DataFrame API และปิดท้ายด้วยการเขียนผลลัพธ์เป็น Lakehouse table เพื่อเห็นความต่างระหว่าง temp view กับ table ที่ persist จริง


## Goal of the day

1. อธิบายได้ว่า **Spark SQL** คืออะไร และใช้ร่วมกับ PySpark DataFrame ได้อย่างไร
2. สร้างและใช้งาน **temporary view** ด้วย `createOrReplaceTempView()` ได้
3. เข้าใจความต่างระหว่าง **temp view**, **global temp view** และ **Lakehouse table**
4. ใช้ `spark.sql()` เพื่อ query, join, aggregate และ create reusable SQL result ได้
5. เปรียบเทียบ logic เดียวกันระหว่าง **SQL API** และ **DataFrame API** ได้


## Why it matters in production

ใน production pipeline หลายทีมไม่ได้ใช้ PySpark DataFrame API อย่างเดียว แต่จะผสม SQL เข้าไปด้วย เพราะ:

- Business logic จำนวนมากอ่านง่ายกว่าในรูปแบบ SQL เช่น `JOIN`, `GROUP BY`, `CASE WHEN`
- Data analyst หรือ BI engineer มักคุ้นกับ SQL มากกว่า PySpark syntax
- Fabric Notebook สามารถทำงานกับทั้ง PySpark และ SQL-style transformation ได้
- Temp view ช่วยให้เราสลับจาก DataFrame ไป SQL ได้เร็ว โดยไม่ต้อง write table ก่อนทุกครั้ง
- การเข้าใจว่า view ไหนเป็น temporary และ table ไหน persist จริง ช่วยลดปัญหา query หายหลัง restart session

Production takeaway:

> Temp view เหมาะกับ intermediate logic ใน notebook/session แต่ถ้าต้องการให้ข้อมูลอยู่ข้าม session หรือให้ pipeline อื่นใช้ต่อ ควร write เป็น Lakehouse table ให้ชัดเจน


## Key concepts

| Concept | Meaning |
|---|---|
| Spark SQL | SQL engine ของ Spark ที่ใช้ query temp view, global temp view หรือ Lakehouse table ได้ |
| `spark.sql()` | ใช้ run SQL statement จาก PySpark และ return ผลลัพธ์เป็น DataFrame |
| Temp View | View ชั่วคราวที่ผูกกับ Spark session ปัจจุบัน สร้างจาก DataFrame ด้วย `createOrReplaceTempView()` |
| Global Temp View | View ชั่วคราวที่อยู่ใน database พิเศษชื่อ `global_temp` และสามารถถูกเรียกใช้ข้าม session ภายใน Spark application เดียวกันได้ |
| Lakehouse Table | Table ที่ persist จริงใน Lakehouse และอ่านกลับได้ด้วย `spark.read.table()` หรือ `spark.sql()` |
| SQL API | เขียน transformation ด้วย SQL string |
| DataFrame API | เขียน transformation ด้วย method chain เช่น `select`, `filter`, `join`, `groupBy` |
| CTE | Common Table Expression หรือ `WITH` clause สำหรับแยก SQL logic เป็น step อ่านง่ายขึ้น |


## Concept explanation

### Spark SQL คืออะไร?

Spark SQL คือส่วนของ Spark ที่ทำให้เราเขียน SQL query บน distributed data ได้ โดยใน PySpark เรามักเรียกผ่าน:

```python
df_result = spark.sql("SELECT * FROM some_view")
```

ผลลัพธ์ที่ได้จะเป็น DataFrame ดังนั้นหลัง query ด้วย SQL แล้ว เราสามารถใช้ DataFrame API ต่อได้ เช่น `.filter()`, `.withColumn()`, `.write` หรือ `.show()`

### Temp View คืออะไร?

Temp view คือชื่อชั่วคราวที่เราตั้งให้ DataFrame เพื่อให้ query ด้วย SQL ได้ เช่น:

```python
df_transactions.createOrReplaceTempView("transactions_vw")
```

จากนั้น query ได้ด้วย:

```python
spark.sql("SELECT * FROM transactions_vw")
```

สิ่งที่ต้องจำคือ temp view ไม่ใช่ table ที่ persist จริง ถ้า session จบหรือ notebook ถูก restart view จะหาย ต้องสร้างใหม่จาก DataFrame อีกครั้ง

### Temp View vs Lakehouse Table

Temp view เหมาะกับ intermediate step ระหว่างทำ notebook เช่น clean, join, aggregate เพื่อทดลอง logic

Lakehouse table เหมาะกับ output ที่ต้องเก็บจริง เช่น curated table, summary table หรือ result ที่ pipeline อื่นต้องอ่านต่อ

> Temp view = ชั่วคราวใน session  
> Lakehouse table = persist ใน Lakehouse

### SQL API vs DataFrame API

ทั้ง SQL API และ DataFrame API สามารถทำ logic เดียวกันได้หลายกรณี เช่น filter, join, groupBy และ aggregate

เลือกใช้ตามโจทย์:

- SQL เหมาะเมื่อ logic เป็น query ใหญ่ อ่านเป็น business rule ได้ง่าย
- DataFrame API เหมาะเมื่อ logic ต้องเขียนแบบ programmatic, reuse function หรือ chain transformation หลาย step
- ใน production ควรเลือก style ที่ทีมอ่านและ maintain ได้ดีที่สุด ไม่จำเป็นต้องยึด style เดียวเสมอไป


## Mermaid diagram: DataFrame, Temp View, SQL, and Lakehouse Table

```mermaid
flowchart LR
    A[PySpark DataFrame] --> B[createOrReplaceTempView]
    B --> C[Temp View in Spark Session]
    C --> D[spark.sql SELECT / JOIN / GROUP BY]
    D --> E[Result DataFrame]
    E --> F[More PySpark transformations]
    E --> G[write.saveAsTable]
    G --> H[Lakehouse Table]
    H --> I[spark.sql or spark.read.table]
```

Key idea:

> Temp view ช่วยให้ DataFrame ถูก query ด้วย SQL ได้ แต่ถ้าต้องการเก็บผลลัพธ์จริง ต้อง write เป็น Lakehouse table


## Setup / imports


In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 3, Finished, Available, Finished, False)

## Create mock data

วันนี้จะใช้ mock data 3 ชุด:

- `df_transactions` = transaction fact data
- `df_customers` = customer master data
- `df_products` = product master data

ข้อมูลตั้งใจให้เล็กพอสำหรับ learning lab แต่ยังมี pattern ใกล้เคียงงานจริง เช่น join, filter, aggregate, status และ source system


In [2]:
transactions_data = [
    ("T001", 101, 1001, "2026-05-01", 1200.50, "success", "web"),
    ("T002", 102, 1002, "2026-05-01", 250.00, "success", "mobile"),
    ("T003", 103, 1003, "2026-05-02", 3400.00, "success", "branch"),
    ("T004", 104, 1001, "2026-05-02", 0.00, "failed", "web"),
    ("T005", 105, 1004, "2026-05-03", 780.00, "pending", "mobile"),
    ("T006", 106, 1002, "2026-05-03", 5600.00, "success", "branch"),
    ("T007", 107, 1005, "2026-05-04", 150.00, "failed", "web"),
    ("T008", 108, 1003, "2026-05-04", 980.00, "success", "mobile"),
    ("T009", 101, 1004, "2026-05-05", 2100.00, "success", "web"),
]

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("product_id", T.IntegerType(), False),
    T.StructField("transaction_date", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
])

customers_data = [
    (101, "Alice", "Bangkok", "active"),
    (102, "Bob", "Bangkok", "active"),
    (103, "Charlie", "Chiang Mai", "active"),
    (104, "Darin", "Rayong", "inactive"),
    (105, "Ekk", "Bangkok", "active"),
    (106, "Fah", "Phuket", "active"),
    (108, "Hana", "Chiang Mai", "active"),
]

customer_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("region", T.StringType(), True),
    T.StructField("customer_status", T.StringType(), True),
])

products_data = [
    (1001, "Starter Package", "subscription"),
    (1002, "Premium Package", "subscription"),
    (1003, "Installation Service", "service"),
    (1004, "Maintenance Service", "service"),
    (1005, "Consulting Add-on", "service"),
]

product_schema = T.StructType([
    T.StructField("product_id", T.IntegerType(), False),
    T.StructField("product_name", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
])

df_transactions_raw = spark.createDataFrame(transactions_data, transaction_schema)
df_transactions = df_transactions_raw.withColumn(
    "transaction_date",
    F.to_date("transaction_date")
)

df_customers = spark.createDataFrame(customers_data, customer_schema)
df_products = spark.createDataFrame(products_data, product_schema)

print("Transactions")
df_transactions.show()
df_transactions.printSchema()

print("Customers")
df_customers.show()

print("Products")
df_products.show()

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 5, Finished, Available, Finished, False)

Transactions
+--------------+-----------+----------+----------------+------+-------+-------------+
|transaction_id|customer_id|product_id|transaction_date|amount| status|source_system|
+--------------+-----------+----------+----------------+------+-------+-------------+
|          T001|        101|      1001|      2026-05-01|1200.5|success|          web|
|          T002|        102|      1002|      2026-05-01| 250.0|success|       mobile|
|          T003|        103|      1003|      2026-05-02|3400.0|success|       branch|
|          T004|        104|      1001|      2026-05-02|   0.0| failed|          web|
|          T005|        105|      1004|      2026-05-03| 780.0|pending|       mobile|
|          T006|        106|      1002|      2026-05-03|5600.0|success|       branch|
|          T007|        107|      1005|      2026-05-04| 150.0| failed|          web|
|          T008|        108|      1003|      2026-05-04| 980.0|success|       mobile|
|          T009|        101|      1004|  

## PySpark code examples

ใน section นี้เราจะเริ่มจาก DataFrame แล้วค่อย register เป็น temp view จากนั้นจะใช้ `spark.sql()` เพื่อ query, join, aggregate และ write result เป็น Lakehouse table


### Example 1: Register DataFrames as temp views

ใช้ `createOrReplaceTempView()` เพื่อสร้างชื่อ view ชั่วคราวใน Spark session ปัจจุบัน

คำว่า `OrReplace` ทำให้ rerun cell ได้ง่ายขึ้น เพราะถ้า view เดิมมีอยู่แล้ว Spark จะ replace ด้วย DataFrame ล่าสุด


In [3]:
df_transactions.createOrReplaceTempView("day10_transactions_vw")
df_customers.createOrReplaceTempView("day10_customers_vw")
df_products.createOrReplaceTempView("day10_products_vw")

print("Temp views created:")
print("- day10_transactions_vw")
print("- day10_customers_vw")
print("- day10_products_vw")

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 7, Finished, Available, Finished, False)

Temp views created:
- day10_transactions_vw
- day10_customers_vw
- day10_products_vw


### Example 2: Query temp view with Spark SQL

หลังจากสร้าง temp view แล้ว เราสามารถใช้ SQL syntax เพื่อ filter, select และสร้าง derived column ได้


In [4]:
df_success_transactions_sql = spark.sql("""
    SELECT
        transaction_id,
        customer_id,
        product_id,
        transaction_date,
        amount,
        source_system,
        CASE
            WHEN amount >= 1000 THEN 'high_value'
            ELSE 'standard'
        END AS amount_band
    FROM day10_transactions_vw
    WHERE status = 'success'
    ORDER BY transaction_date, transaction_id
""")

df_success_transactions_sql.show()

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 9, Finished, Available, Finished, False)

+--------------+-----------+----------+----------------+------+-------------+-----------+
|transaction_id|customer_id|product_id|transaction_date|amount|source_system|amount_band|
+--------------+-----------+----------+----------------+------+-------------+-----------+
|          T001|        101|      1001|      2026-05-01|1200.5|          web| high_value|
|          T002|        102|      1002|      2026-05-01| 250.0|       mobile|   standard|
|          T003|        103|      1003|      2026-05-02|3400.0|       branch| high_value|
|          T006|        106|      1002|      2026-05-03|5600.0|       branch| high_value|
|          T008|        108|      1003|      2026-05-04| 980.0|       mobile|   standard|
|          T009|        101|      1004|      2026-05-05|2100.0|          web| high_value|
+--------------+-----------+----------+----------------+------+-------------+-----------+



### Example 3: Join and aggregate using SQL

SQL มักอ่านง่ายเมื่อ logic เป็น business-style query เช่น join fact table กับ master data แล้วสรุปยอดตาม region และ product category


In [5]:
df_region_category_summary_sql = spark.sql("""
    SELECT
        c.region,
        p.category,
        COUNT(*) AS success_transaction_count,
        ROUND(SUM(t.amount), 2) AS total_success_amount,
        ROUND(AVG(t.amount), 2) AS avg_success_amount
    FROM day10_transactions_vw t
    INNER JOIN day10_customers_vw c
        ON t.customer_id = c.customer_id
    INNER JOIN day10_products_vw p
        ON t.product_id = p.product_id
    WHERE t.status = 'success'
      AND c.customer_status = 'active'
    GROUP BY
        c.region,
        p.category
    ORDER BY
        total_success_amount DESC
""")

df_region_category_summary_sql.show()

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 11, Finished, Available, Finished, False)

+----------+------------+-------------------------+--------------------+------------------+
|    region|    category|success_transaction_count|total_success_amount|avg_success_amount|
+----------+------------+-------------------------+--------------------+------------------+
|    Phuket|subscription|                        1|              5600.0|            5600.0|
|Chiang Mai|     service|                        2|              4380.0|            2190.0|
|   Bangkok|     service|                        1|              2100.0|            2100.0|
|   Bangkok|subscription|                        2|              1450.5|            725.25|
+----------+------------+-------------------------+--------------------+------------------+



### Example 4: Reproduce the same logic using DataFrame API

ตัวอย่างนี้ทำ logic ใกล้เคียงกับ Example 3 แต่เขียนด้วย DataFrame API เพื่อให้เห็นว่า SQL และ DataFrame API สามารถแทนกันได้ในหลายกรณี


In [6]:
df_region_category_summary_df_api = (
    df_transactions.alias("t")
    .join(df_customers.alias("c"), F.col("t.customer_id") == F.col("c.customer_id"), "inner")
    .join(df_products.alias("p"), F.col("t.product_id") == F.col("p.product_id"), "inner")
    .filter(F.col("t.status") == "success")
    .filter(F.col("c.customer_status") == "active")
    .groupBy(
        F.col("c.region").alias("region"),
        F.col("p.category").alias("category")
    )
    .agg(
        F.count("*").alias("success_transaction_count"),
        F.round(F.sum("t.amount"), 2).alias("total_success_amount"),
        F.round(F.avg("t.amount"), 2).alias("avg_success_amount")
    )
    .orderBy(F.desc("total_success_amount"))
)

df_region_category_summary_df_api.show()

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 14, Finished, Available, Finished, False)

+----------+------------+-------------------------+--------------------+------------------+
|    region|    category|success_transaction_count|total_success_amount|avg_success_amount|
+----------+------------+-------------------------+--------------------+------------------+
|    Phuket|subscription|                        1|              5600.0|            5600.0|
|Chiang Mai|     service|                        2|              4380.0|            2190.0|
|   Bangkok|     service|                        1|              2100.0|            2100.0|
|   Bangkok|subscription|                        2|              1450.5|            725.25|
+----------+------------+-------------------------+--------------------+------------------+



### Example 5: Create a reusable temp view from SQL

นอกจากสร้าง temp view จาก DataFrame แล้ว เราสามารถสร้าง temp view ด้วย SQL ได้เช่นกัน เหมาะกับการแยก intermediate result ให้ query ต่อได้ง่าย


In [7]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW day10_success_enriched_vw AS
    SELECT
        t.transaction_id,
        t.customer_id,
        c.customer_name,
        c.region,
        t.product_id,
        p.product_name,
        p.category,
        t.transaction_date,
        t.amount,
        t.source_system
    FROM day10_transactions_vw t
    INNER JOIN day10_customers_vw c
        ON t.customer_id = c.customer_id
    INNER JOIN day10_products_vw p
        ON t.product_id = p.product_id
    WHERE t.status = 'success'
      AND c.customer_status = 'active'
""")

spark.sql("""
    SELECT
        region,
        category,
        COUNT(*) AS record_count,
        ROUND(SUM(amount), 2) AS total_amount
    FROM day10_success_enriched_vw
    GROUP BY region, category
    ORDER BY total_amount DESC
""").show()

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 17, Finished, Available, Finished, False)

+----------+------------+------------+------------+
|    region|    category|record_count|total_amount|
+----------+------------+------------+------------+
|    Phuket|subscription|           1|      5600.0|
|Chiang Mai|     service|           2|      4380.0|
|   Bangkok|     service|           1|      2100.0|
|   Bangkok|subscription|           2|      1450.5|
+----------+------------+------------+------------+



### Example 6: Create and query a global temp view

Global temp view จะอยู่ใน database พิเศษชื่อ `global_temp` เสมอ ดังนั้นเวลา query ต้องเขียนชื่อเต็ม เช่น `global_temp.view_name`

ในการใช้งานจริง ไม่ควรใช้ global temp view โดยไม่จำเป็น เพราะมันมองเห็นได้กว้างกว่า temp view ปกติภายใน Spark application เดียวกัน ทำให้ trace ได้ยากขึ้นว่า view นี้ถูกสร้างจาก session ไหน และใช้เพื่ออะไร


In [8]:
df_success_transactions_sql.createOrReplaceGlobalTempView("day10_success_transactions_global_vw")

spark.sql("""
    SELECT
        source_system,
        COUNT(*) AS success_transaction_count,
        ROUND(SUM(amount), 2) AS total_success_amount
    FROM global_temp.day10_success_transactions_global_vw
    GROUP BY source_system
    ORDER BY total_success_amount DESC
""").show()

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 18, Finished, Available, Finished, False)

+-------------+-------------------------+--------------------+
|source_system|success_transaction_count|total_success_amount|
+-------------+-------------------------+--------------------+
|       branch|                        2|              9000.0|
|          web|                        2|              3300.5|
|       mobile|                        2|              1230.0|
+-------------+-------------------------+--------------------+



### Example 7: Continue PySpark transformations after SQL

ผลลัพธ์จาก `spark.sql()` ยังเป็น DataFrame ดังนั้นสามารถใช้ PySpark transformation ต่อได้ตามปกติ


In [9]:
df_high_region_sql = spark.sql("""
    SELECT
        region,
        SUM(amount) AS total_amount
    FROM day10_success_enriched_vw
    GROUP BY region
""")

df_high_region_with_flag = (
    df_high_region_sql
    .withColumn(
        "region_size",
        F.when(F.col("total_amount") >= 3000, F.lit("large"))
         .otherwise(F.lit("standard"))
    )
    .orderBy(F.desc("total_amount"))
)

df_high_region_with_flag.show()

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 19, Finished, Available, Finished, False)

+----------+------------+-----------+
|    region|total_amount|region_size|
+----------+------------+-----------+
|    Phuket|      5600.0|      large|
|Chiang Mai|      4380.0|      large|
|   Bangkok|      3550.5|      large|
+----------+------------+-----------+



### Example 8: Write SQL result to a Lakehouse table

Temp view จะหายเมื่อ session จบ แต่ Lakehouse table จะถูก persist ไว้ใน Lakehouse

ใน Microsoft Fabric Notebook ให้แน่ใจว่า notebook attach Lakehouse แล้วก่อน run cell นี้


In [11]:
(
    df_region_category_summary_sql
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("day10_region_category_summary")
)

spark.sql("""
    SELECT *
    FROM day10_region_category_summary
    ORDER BY total_success_amount DESC
""").show()

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 25, Finished, Available, Finished, False)

+----------+------------+-------------------------+--------------------+------------------+
|    region|    category|success_transaction_count|total_success_amount|avg_success_amount|
+----------+------------+-------------------------+--------------------+------------------+
|    Phuket|subscription|                        1|              5600.0|            5600.0|
|Chiang Mai|     service|                        2|              4380.0|            2190.0|
|   Bangkok|     service|                        1|              2100.0|            2100.0|
|   Bangkok|subscription|                        2|              1450.5|            725.25|
+----------+------------+-------------------------+--------------------+------------------+



## Quick recap

| Question | Answer |
|---|---|
| Spark SQL ใช้ทำอะไร? | ใช้ query temp view, global temp view หรือ table ด้วย SQL syntax |
| Temp view อยู่ถาวรไหม? | ไม่ถาวร อยู่ใน Spark session ปัจจุบัน |
| สร้าง temp view จาก DataFrame ใช้อะไร? | `createOrReplaceTempView()` |
| Query SQL จาก PySpark ใช้อะไร? | `spark.sql("SELECT ...")` |
| Global temp view query ยังไง? | ต้องอ้างอิงผ่าน `global_temp.<view_name>` |
| ถ้าต้องการเก็บผลลัพธ์จริงควรทำอะไร? | write เป็น Lakehouse table เช่น `saveAsTable()` |


## Exercises


### Exercise 1: Create a SQL result for Bangkok success transactions

สร้าง DataFrame ชื่อ `df_bangkok_success_sql` ด้วย `spark.sql()` จาก temp view `day10_transactions_vw`

Requirements:

- filter เฉพาะ `status = 'success'`
- filter เฉพาะ customer ที่อยู่ region `Bangkok` โดย join กับ `day10_customers_vw`
- select columns:
  - `transaction_id`
  - `customer_id`
  - `customer_name`
  - `region`
  - `amount`
  - `transaction_date`
- sort by `transaction_date`, `transaction_id`

Expected result:

- ได้เฉพาะ successful transactions ของลูกค้าใน Bangkok
- ใช้ `.show()` เพื่อตรวจ output


In [12]:
df_bangkok_success_sql = spark.sql("""
    SELECT
        t.transaction_id,
        t.customer_id,
        c.customer_name,
        c.region,
        t.amount,
        t.transaction_date
    FROM day10_transactions_vw t
    INNER JOIN day10_customers_vw c
        ON t.customer_id = c.customer_id
    WHERE t.status = 'success'
      AND c.region = 'Bangkok'
    ORDER BY t.transaction_date, t.transaction_id
""")

df_bangkok_success_sql.show()

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 26, Finished, Available, Finished, False)

+--------------+-----------+-------------+-------+------+----------------+
|transaction_id|customer_id|customer_name| region|amount|transaction_date|
+--------------+-----------+-------------+-------+------+----------------+
|          T001|        101|        Alice|Bangkok|1200.5|      2026-05-01|
|          T002|        102|          Bob|Bangkok| 250.0|      2026-05-01|
|          T009|        101|        Alice|Bangkok|2100.0|      2026-05-05|
+--------------+-----------+-------------+-------+------+----------------+



### Exercise 2: Create an enriched temp view using SQL

สร้าง temp view ชื่อ `day10_transaction_detail_vw` ด้วย SQL

Requirements:

- join transactions, customers และ products
- include เฉพาะ records ที่ `status = 'success'`
- columns ที่ควรมี:
  - `transaction_id`
  - `customer_name`
  - `region`
  - `product_name`
  - `category`
  - `amount`
  - `source_system`
- query temp view แล้วแสดงผลลัพธ์

Expected result:

- ได้ transaction detail ที่อ่านง่ายขึ้น เพราะมี customer และ product context เพิ่มเข้ามา


In [13]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW day10_transaction_detail_vw AS
    SELECT
        t.transaction_id,
        c.customer_name,
        c.region,
        p.product_name,
        p.category,
        t.amount,
        t.source_system
    FROM day10_transactions_vw t
    INNER JOIN day10_customers_vw c
        ON t.customer_id = c.customer_id
    INNER JOIN day10_products_vw p
        ON t.product_id = p.product_id
    WHERE t.status = 'success'
""")

df_transaction_detail = spark.sql("""
    SELECT *
    FROM day10_transaction_detail_vw
    ORDER BY amount DESC
""")

df_transaction_detail.show()

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 28, Finished, Available, Finished, False)

+--------------+-------------+----------+--------------------+------------+------+-------------+
|transaction_id|customer_name|    region|        product_name|    category|amount|source_system|
+--------------+-------------+----------+--------------------+------------+------+-------------+
|          T006|          Fah|    Phuket|     Premium Package|subscription|5600.0|       branch|
|          T003|      Charlie|Chiang Mai|Installation Service|     service|3400.0|       branch|
|          T009|        Alice|   Bangkok| Maintenance Service|     service|2100.0|          web|
|          T001|        Alice|   Bangkok|     Starter Package|subscription|1200.5|          web|
|          T008|         Hana|Chiang Mai|Installation Service|     service| 980.0|       mobile|
|          T002|          Bob|   Bangkok|     Premium Package|subscription| 250.0|       mobile|
+--------------+-------------+----------+--------------------+------------+------+-------------+



### Exercise 3: Compare SQL result with DataFrame API and write table

สร้าง summary ตาม `source_system` ด้วย DataFrame API จาก `df_transaction_detail`

Requirements:

- group by `source_system`
- calculate:
  - `transaction_count`
  - `total_amount`
  - `avg_amount`
- order by `total_amount` descending
- write เป็น Lakehouse table ชื่อ `day10_exercise_source_system_summary`
- read table กลับด้วย `spark.sql()`

Expected result:

- ได้ summary table ตาม source system
- เห็นว่า SQL result สามารถนำไป transform ต่อด้วย DataFrame API และ persist เป็น Lakehouse table ได้


In [14]:
df_source_system_summary = (
    df_transaction_detail
    .groupBy("source_system")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount")
    )
    .orderBy(F.desc("total_amount"))
)

df_source_system_summary.show()

(
    df_source_system_summary.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("day10_exercise_source_system_summary")
)

spark.sql("""
    SELECT *
    FROM day10_exercise_source_system_summary
    ORDER BY total_amount DESC
""").show()

StatementMeta(, 84d4f58e-0a18-49cb-9e08-f91e95b6f6ab, 41, Finished, Available, Finished, False)

+-------------+-----------------+------------+----------+
|source_system|transaction_count|total_amount|avg_amount|
+-------------+-----------------+------------+----------+
|       branch|                2|      9000.0|    4500.0|
|          web|                2|      3300.5|   1650.25|
|       mobile|                2|      1230.0|     615.0|
+-------------+-----------------+------------+----------+

+-------------+-----------------+------------+----------+
|source_system|transaction_count|total_amount|avg_amount|
+-------------+-----------------+------------+----------+
|       branch|                2|      9000.0|    4500.0|
|          web|                2|      3300.5|   1650.25|
|       mobile|                2|      1230.0|     615.0|
+-------------+-----------------+------------+----------+



## Common mistakes

1. **คิดว่า temp view คือ table ถาวร**

   Temp view อยู่แค่ใน Spark session ถ้า restart notebook หรือ session หาย ต้องสร้าง view ใหม่

2. **แก้ DataFrame แล้วลืม register temp view ใหม่**

   ถ้า DataFrame logic เปลี่ยน แต่ temp view เดิมยังชี้ logic เก่าอยู่ ผลลัพธ์ SQL อาจไม่ตรงกับที่คิด ควรใช้ `createOrReplaceTempView()` หลังเตรียม DataFrame ล่าสุด

3. **ลืมอ้างอิง `global_temp` ตอน query global temp view**

   Global temp view ต้อง query ผ่าน `global_temp.<view_name>` ไม่ใช่ชื่อ view ตรง ๆ

4. **สับสนระหว่าง SQL string กับ DataFrame result**

   `spark.sql("SELECT ...")` จะ return เป็น DataFrame ดังนั้นถ้าต้องการเห็น output ยังต้องใช้ action เช่น `.show()` หรือ `.count()`

5. **ใช้ SQL ยาวเกินไปโดยไม่แยก step**

   SQL ที่ยาวมากอาจ debug ยาก ควรแยกด้วย CTE หรือ temp view intermediate เมื่อ logic ซับซ้อนขึ้น

6. **ใช้ `SELECT *` ใน production logic โดยไม่ตั้งใจ**

   `SELECT *` สะดวกตอน explore data แต่ใน pipeline จริงควรเลือก columns ชัดเจนเพื่อลดปัญหา schema เปลี่ยนแล้วกระทบ downstream

7. **เขียน dynamic SQL จาก input โดยไม่ระวัง**

   ถ้าต้องประกอบ SQL string จาก parameter ควรควบคุม value ให้ปลอดภัยและชัดเจน ไม่ปล่อยให้ชื่อ table, column หรือ filter value มาจาก input ที่ไม่น่าเชื่อถือโดยตรง


## Expected Output / Success Criteria

เมื่อจบ Day 10 ควรทำได้:

- อธิบายได้ว่า **Spark SQL** คือ SQL engine ที่ใช้ query temp view, global temp view หรือ Lakehouse table ได้
- อธิบายได้ว่า DataFrame ต้องถูก register เป็น temp view / global temp view หรือถูก write เป็น table ก่อน จึงจะถูกอ้างอิงผ่าน `spark.sql()` ได้
- สร้าง temp view จาก DataFrame ด้วย `createOrReplaceTempView()` ได้
- ใช้ SQL syntax เช่น `SELECT`, `WHERE`, `CASE WHEN`, `JOIN`, `GROUP BY`, `ORDER BY` ได้
- สร้าง reusable temp view จาก SQL ด้วย `CREATE OR REPLACE TEMP VIEW ... AS SELECT ...` ได้
- อธิบายได้ว่า temp view เป็น session-scoped และไม่ใช่ Lakehouse table ถาวร
- อธิบายได้ว่า global temp view ต้อง query ผ่าน `global_temp.<view_name>`
- เปรียบเทียบ logic เดียวกันระหว่าง SQL API และ DataFrame API ได้
- นำผลลัพธ์จาก `spark.sql()` ไป transform ต่อด้วย DataFrame API ได้
- Write SQL result เป็น Lakehouse table และอ่านกลับด้วย `spark.sql()` ได้ ถ้า attach Lakehouse เรียบร้อย


## Final takeaway

Spark SQL เป็นเครื่องมือสำคัญที่ช่วยให้ PySpark pipeline อ่านง่ายขึ้นในบางโจทย์ โดยเฉพาะ logic แบบ query, join และ aggregation

> DataFrame API และ SQL API ไม่ใช่คู่แข่งกัน แต่เป็นสองวิธีในการอธิบาย transformation ให้ Spark เข้าใจ

สำหรับ Data Engineer สิ่งที่ต้องจำคือ:

- ใช้ temp view เพื่อสลับจาก DataFrame ไป SQL ระหว่างทำ notebook หรือ intermediate logic
- อย่าคิดว่า temp view เป็น table ถาวร
- ถ้าต้องให้ pipeline อื่นใช้ต่อ ให้ write เป็น Lakehouse table
- เลือก SQL หรือ DataFrame API ตาม readability, maintainability และ team convention
- ตรวจ output ด้วย `.show()`, `.count()` หรือ query validation ก่อน write table จริง


## Optional cleanup

In [ ]:
# spark.catalog.dropTempView("day10_transactions_vw")
# spark.catalog.dropTempView("day10_customers_vw")
# spark.catalog.dropTempView("day10_products_vw")
# spark.catalog.dropTempView("day10_success_enriched_vw")
# spark.catalog.dropTempView("day10_transaction_detail_vw")
# spark.catalog.dropGlobalTempView("day10_success_transactions_global_vw")

# spark.sql("DROP TABLE IF EXISTS day10_region_category_summary")
# spark.sql("DROP TABLE IF EXISTS day10_exercise_source_system_summary")